In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch02-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "02-01-read-write-draw.py"

# 結構與溶解度寫入 SDF 後，還對得起來嗎？

修改第三格後，重跑第三格與第四格。首次載入 ESOL 需要網路；transformers=[]，溶解度未經標準化。


In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
from rdkit import Chem
from rdkit.Chem import Draw, rdDepictor
import deepchem as dc

# ESOL：1,128 筆水溶解度；log10(mol/L)。DOI: 10.1021/ci034243x
# 本節只核對結構和欄位，不訓練模型；保留 scaffold 切分再合併讀取全部資料。
_, parts, _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
records = []
for part in parts:
    for smiles, y in zip(part.ids, part.y):
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            raise ValueError(f"ESOL 結構無法讀入：{smiles}")
        records.append((mol, float(y[0])))
# 下面三個字串只作查詢鍵；圖和溶解度皆取自 ESOL 實際資料列。
wanted = [("Phenol", "Oc1ccccc1"), ("Anisole", "COc1ccccc1"),
          ("Cyclohexanol", "OC1CCCCC1")]
selected = []
for name, key in wanted:
    hits = [(m, y) for m, y in records if Chem.MolToSmiles(m) == key]
    assert len(hits) == 1, (name, len(hits))
    mol, value = hits[0]
    selected.append((name, mol, value))
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)

# 寫入自己的暫存檔，再核對結構和量測欄位是否保留。
with tempfile.TemporaryDirectory() as folder:
    sdf_path = Path(folder) / "selected.sdf"
    with Chem.SDWriter(str(sdf_path)) as writer:
        for name, mol, value in selected:
            mol.SetProp("_Name", name)
            mol.SetProp("logS_mol_L", str(value))
            rdDepictor.Compute2DCoords(mol)
            writer.write(mol)
    restored = list(Chem.SDMolSupplier(str(sdf_path)))
    assert len(restored) == len(selected)
    for (name, original, value), mol in zip(selected, restored):
        assert mol is not None
        same_structure = Chem.MolToSmiles(original) == Chem.MolToSmiles(mol)
        same_value = float(mol.GetProp("logS_mol_L")) == value
        assert same_structure and same_value
        print(name, Chem.MolToSmiles(mol), value, same_structure, same_value)
    # Molfile 適合單一結構；本例不靠它保存一般 SDF 性質欄位。
    mol_path = Path(folder) / "phenol.mol"
    Chem.MolToMolFile(restored[0], str(mol_path))
    single = Chem.MolFromMolFile(str(mol_path))
    assert single is not None
    assert Chem.MolToSmiles(single) == Chem.MolToSmiles(restored[0])
    panel = Draw.MolsToGridImage(restored, molsPerRow=3, subImgSize=(330, 260),
        legends=[f"{name}\nlogS = {value:g}" for name, _, value in selected])
    panel.save(out / "02-01-read-write-draw.png")
print("ESOL rows:", len(records), "SDF records:", len(restored))


In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.replace("This module requires PyTorch to be installed.\n", "").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "02-01-read-write-draw.png")))